# K-Means Clustering

This notebook applies K-Means to the shared TF-IDF review features to discover groups of similar customer reviews. K-Means is an unsupervised model, so it is evaluated with clustering metrics rather than Accuracy/Precision/Recall/F1.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from scipy.sparse import load_npz, vstack
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

In [ ]:
# Load the same shared TF-IDF features prepared by Member 2
X_train = load_npz('X_train_tfidf.npz')
X_test = load_npz('X_test_tfidf.npz')

# Clustering is unsupervised, so we can cluster all review feature vectors
X_all = vstack([X_train, X_test])

print('All TF-IDF reviews shape:', X_all.shape)

## Elbow Method

Test several values of K and inspect inertia to help choose a reasonable number of clusters.

In [ ]:
k_values = range(2, 9)
inertias = []

for k in k_values:
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    model.fit(X_all)
    inertias.append(model.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(list(k_values), inertias, marker='o')
plt.title('Elbow Method for K-Means')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Inertia')
plt.show()

## Train K-Means

Start with 3 clusters. You can change `n_clusters` after inspecting the Elbow Method.

In [ ]:
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(X_all)

print('Number of clusters:', kmeans.n_clusters)
print('Inertia:', kmeans.inertia_)

In [ ]:
# Silhouette Score can be expensive, so use a reproducible sample
sample_size = min(5000, X_all.shape[0])
silhouette = silhouette_score(
    X_all,
    cluster_labels,
    sample_size=sample_size,
    random_state=42
)

print('Silhouette Score:', silhouette)

In [ ]:
cluster_counts = pd.Series(cluster_labels).value_counts().sort_index()
cluster_counts.index = [f'Cluster {i}' for i in cluster_counts.index]

print(cluster_counts)

cluster_counts.plot(kind='bar', figsize=(8, 5))
plt.title('Number of Reviews in Each Cluster')
plt.xlabel('Cluster')
plt.ylabel('Number of Reviews')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## Interpretation

Cluster IDs (0, 1, 2, ...) do **not** automatically mean Complaint, Positive Feedback, or another business category. To give each cluster a meaningful name, inspect the most important TF-IDF terms and sample reviews belonging to each cluster.

In [ ]:
import joblib

tfidf = joblib.load('tfidf_vectorizer.pkl')
terms = tfidf.get_feature_names_out()
order_centroids = kmeans.cluster_centers_.argsort()[:, ::-1]

for cluster_id in range(kmeans.n_clusters):
    top_terms = [terms[i] for i in order_centroids[cluster_id, :10]]
    print(f'Cluster {cluster_id} top terms:', ', '.join(top_terms))